# 02 · Cleaning — Deals

**Goal:** clean the Deals table — fix types, handle missing values, flag anomalies, recover `Payment Type`, and build derived columns for analysis.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import time, timedelta
import re

## 1. Load & set types

Load Deals; set `Id` and `Contact Name` as strings, parse `Created Time` and `Closing Date` as datetime.

In [ ]:
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

deals = pd.read_excel(DATA_RAW / 'Deals (Done).xlsx', dtype={'Id':str, 'Contact Name': str})
deals['Created Time'] = pd.to_datetime(deals['Created Time'], format='%d.%m.%Y %H:%M')
deals['Closing Date'] = pd.to_datetime(deals['Closing Date'], format='%d.%m.%Y')
print(f"Initial: {deals.shape}")

Исходно: (21595, 23)


## 2. Initial inspection

Initial review of types, filters, and missing values was done in Excel. Here we check shape, types, missing values, and unique counts per column.

In [ ]:
deals.shape # initial (21595, 23)

(21595, 23)

In [ ]:
deals.dtypes 

Id                                str
Deal Owner Name                   str
Closing Date           datetime64[us]
Quality                           str
Stage                             str
Lost Reason                       str
Page                              str
Campaign                          str
SLA                            object
Content                           str
Term                              str
Source                            str
Payment Type                      str
Product                           str
Education Type                    str
Created Time           datetime64[us]
Course duration               float64
Months of study               float64
Initial Amount Paid            object
Offer Total Amount             object
Contact Name                      str
City                              str
Level of Deutsch               object
dtype: object

In [ ]:
deals.isnull().sum() # missing values

Id                         2
Deal Owner Name           31
Closing Date            6950
Quality                 2255
Stage                      2
Lost Reason             5471
Page                       2
Campaign                5528
SLA                     6062
Content                 7448
Term                    9141
Source                     2
Payment Type           21099
Product                18003
Education Type         18295
Created Time               2
Course duration        18008
Months of study        20755
Initial Amount Paid    17430
Offer Total Amount     17410
Contact Name              63
City                   19084
Level of Deutsch       20344
dtype: int64

The methodology involves going through each column, filling in missing values ​​where possible, removing noise and duplicates, and changing data types.

In [ ]:
deals.nunique() # unique values ​​for each column

Id                     21593
Deal Owner Name           27
Closing Date             359
Quality                    6
Stage                     13
Lost Reason               21
Page                      34
Campaign                 154
SLA                    13357
Content                  187
Term                     220
Source                    13
Payment Type               3
Product                    5
Education Type             3
Created Time           20334
Course duration            2
Months of study           12
Initial Amount Paid       24
Offer Total Amount        21
Contact Name           18089
City                     876
Level of Deutsch         215
dtype: int64

## 3. Missing-value strategy

Reviewed missing values column by column. Key decisions:

- `Id` — 2 missing, rest unique → drop the 2 empty rows.
- `Deal Owner Name` — 31 missing → fill with "Unknown manager".
- `Closing Date` — ~6,950 missing = open (unclosed) deals → keep as is.
- `Payment Type` — almost all missing → recover partially from payment amount and deal stage (see §4).
- `Course duration` / `Education Type` — fill partially based on product.
- `Initial Amount Paid` / `Offer Total Amount` — convert from European format to EUR; fill for closed deals.
- Low-impact columns (`Page`, `Source`, `Created Time`, a few each) → leave as is, statistically negligible.

In [ ]:
# How many rows have NaN for ALL values?
fully_empty = deals.isna().all(axis=1).sum()
print(f"Fully empty rows: {fully_empty}")

Полностью пустых строк: 1


In [ ]:
deals[deals['Id'].isnull()]  # We have two clearly completely empty lines — delete them.

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
21593,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21594,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,#REF!,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
deals = deals.dropna(subset=['Id']) # Initial (21595, 23)
deals.shape

(21593, 23)

In [10]:
deals.isnull().sum() # number of missing values, table for updating

Id                         0
Deal Owner Name           29
Closing Date            6948
Quality                 2253
Stage                      0
Lost Reason             5469
Page                       0
Campaign                5526
SLA                     6060
Content                 7446
Term                    9139
Source                     0
Payment Type           21097
Product                18001
Education Type         18294
Created Time               0
Course duration        18006
Months of study        20753
Initial Amount Paid    17428
Offer Total Amount     17408
Contact Name              61
City                   19082
Level of Deutsch       20342
dtype: int64

## 4. Column cleaning

### 4.1 Amounts → EUR

`Initial Amount Paid` and `Offer Total Amount` contain currency symbols and European number format. Normalize to float (EUR).

In [11]:
deals['Initial Amount Paid'].astype(str).value_counts().head(50)

Initial Amount Paid
1000          2623
0              876
300            188
500             94
350             82
2000            58
11000           38
200             31
11500           25
3500            22
€ 3.500,00      16
1500            16
450             16
5000            14
4000            13
100             12
3000            11
4500            10
400             10
1                3
600              3
1200             2
700              1
9                1
Name: count, dtype: int64

In [12]:
deals['Offer Total Amount'].astype(str).value_counts().head(30)

Offer Total Amount
11000         1860
0              848
11500          394
5000           295
4000           252
3500           133
9000           115
2500            70
2000            63
3000            58
4500            57
€ 2.900,00      20
1200             6
1000             3
1500             3
10000            2
1                2
6500             1
€ 11398,00       1
11111            1
6000             1
Name: count, dtype: int64

We can see the euro sign and dots in the data. Let's write a function for them and apply it to both columns.

In [ ]:
def normalize_eur(series: pd.Series) -> pd.Series:
    """Values like `€ 3.500,00` and `1000` → cleaned to float via `normalize_eur()`."""
    cleaned = (
        series.astype(str)
        .str.replace('€', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    return pd.to_numeric(cleaned, errors='coerce')

deals['Initial Amount Paid'] = normalize_eur(deals['Initial Amount Paid'])
deals['Offer Total Amount'] = normalize_eur(deals['Offer Total Amount'])

In [14]:
deals['Offer Total Amount'].astype(str).value_counts().head(30)

Offer Total Amount
11000.0    1860
0.0         848
11500.0     394
5000.0      295
4000.0      252
3500.0      133
9000.0      115
2500.0       70
2000.0       63
3000.0       58
4500.0       57
2900.0       20
1200.0        6
1000.0        3
1500.0        3
10000.0       2
1.0           2
6500.0        1
11398.0       1
11111.0       1
6000.0        1
Name: count, dtype: int64

In [ ]:
deals[['Initial Amount Paid', 'Offer Total Amount']].dtypes #check

Initial Amount Paid    float64
Offer Total Amount     float64
dtype: object

### 4.2 SLA → timedelta

`SLA` is stored as object with mixed types. Unify to `timedelta`.

In [ ]:
# What is the type of each value?
print(deals['SLA'].apply(type).value_counts())

SLA
<class 'datetime.time'>         13672
<class 'float'>                  6060
<class 'datetime.timedelta'>     1861
Name: count, dtype: int64


We see three different data types in the column; we convert both types to a uniform `timedelta` format and then apply `pd.to_timedelta` for final unification.

In [ ]:
def to_timedelta_safe(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, time):
        return timedelta(hours=value.hour, minutes=value.minute, seconds=value.second)
    if isinstance(value, timedelta):
        return value
    return pd.NaT  # for all unexpected cases

In [ ]:
deals['SLA'] = deals['SLA'].apply(to_timedelta_safe)
deals['SLA'] = pd.to_timedelta(deals['SLA'])  # Final unification of pandas Timedelta

In [19]:
print(deals['SLA'].dtype)
print(deals['SLA'].describe())
print(f"NaN: {deals['SLA'].isna().sum()}")

timedelta64[us]
count                     15533
mean     1 days 08:10:26.223330
std      8 days 12:47:32.635614
min             0 days 00:00:03
25%             0 days 01:13:00
50%             0 days 05:31:34
75%             0 days 15:38:38
max           311 days 10:34:24
Name: SLA, dtype: object
NaN: 6060


In [20]:
sla_30_days = pd.Timedelta(days=30)
anomaly_count = (deals['SLA'] > sla_30_days).sum()
valid_count = deals['SLA'].notna().sum()
print(f"SLA > 30 days: {anomaly_count}")
print(f"Total filled: {valid_count}")
print(f"Anomaly share: {anomaly_count / valid_count * 100:.2f}%")
print(f"\nMax: {deals['SLA'].max()}")
print(f"Median: {deals['SLA'].median()}")

SLA > 30 дней: 91
Всего заполнено: 15533
Доля аномалий: 0.59%

Максимум: 311 days 10:34:24
Медиана: 0 days 05:31:34


**Note:** SLA ranges from instant response (~seconds) to 311 days. Median ~5.5h. Anomalies (>30 days: 91 rows, 0.59%) flagged for outlier review later.

In [ ]:
deals.dtypes # column types final - all good.

Id                                 str
Deal Owner Name                    str
Closing Date            datetime64[us]
Quality                            str
Stage                              str
Lost Reason                        str
Page                               str
Campaign                           str
SLA                    timedelta64[us]
Content                            str
Term                               str
Source                             str
Payment Type                       str
Product                            str
Education Type                     str
Created Time            datetime64[us]
Course duration                float64
Months of study                float64
Initial Amount Paid            float64
Offer Total Amount             float64
Contact Name                       str
City                               str
Level of Deutsch                object
dtype: object

In [22]:
print(deals['Level of Deutsch'].dtype)
print(deals['Level of Deutsch'].value_counts(dropna=False).head(20))

object
Level of Deutsch
NaN      20342
B1         219
б1         118
в1         100
b1          93
Б1          93
В1          63
А2          53
B2          45
а2          32
б2          30
Б2          20
в2          20
в1-в2       19
b2          19
A2          18
?           15
В2          14
с1          13
А1          11
Name: count, dtype: int64


### 4.3 Categorical cleaning

Review unique values across categorical columns.

In [23]:
categorical_cols = [
    'Stage', 'Quality', 'Lost Reason',
    'Source', 'Campaign', 'Product',
    'Education Type', 'Payment Type',
    'City', 'Level of Deutsch',
    'Deal Owner Name'
]

for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(deals[col].value_counts(dropna=False))


=== Stage ===
Stage
Lost                         15743
Call Delayed                  2248
Registered on Webinar         2072
Payment Done                   858
Waiting For Payment            325
Qualificated                   128
Registered on Offline Day      100
Need to Call - Sales            33
Need To Call                    31
Test Sent                       25
Need a consultation             23
New Lead                         6
Free Education                   1
Name: count, dtype: int64

=== Quality ===
Quality
E - Non Qualified    7634
D - Non Target       6248
C - Low              3459
NaN                  2253
B - Medium           1564
A - High              432
F                       3
Name: count, dtype: int64

=== Lost Reason ===
Lost Reason
NaN                                        5469
Doesn't Answer                             4135
Changed Decision                           2146
Duplicate                                  1771
Non target                              

Findings:

- **Stage** — 13 values → grouped into Won (Payment Done, 858), Lost (15,743), In Progress (~5,000). Base conversion Won/(Won+Lost) = **5.17%**.
- **Level of Deutsch** — 216 raw values across 1,251 filled rows, 94% missing → normalize via regex; treat "German level vs conversion" as indicative only.
- **Source** — 159 rows = `Test` → check before dropping.
- **City** — 348 rows = `-` → encoded missing, replace with NaN.
- **Quality** — 3 rows = `F` → likely typo / legacy category, replace with NaN.
- **Lost Reason** — 22 reasons → grouped: contact issues (~9,000), product objections (~2,500), client mismatch (~1,000), Gutschein refusal & financial (~300). Rich material for business recommendations.

In [24]:
# Replace dash with NaN in City
deals['City'] = deals['City'].replace('-', np.nan)

In [25]:
deals['Quality'] = deals['Quality'].replace('F', np.nan)


In [26]:
categorical_cols = ['City', 'Quality']

for col in categorical_cols:
      print(deals[col].value_counts(dropna=False))

City
NaN                19430
Berlin               182
München               74
Hamburg               62
Nürnberg              45
                   ...  
Bad Gandersheim        1
Braubach               1
Geldern                1
Bassum                 1
Brake                  1
Name: count, Length: 876, dtype: int64
Quality
E - Non Qualified    7634
D - Non Target       6248
C - Low              3459
NaN                  2256
B - Medium           1564
A - High              432
Name: count, dtype: int64


In [27]:
test_leads = deals[deals['Source'] == 'Test']
print(f"Total leads with Source=Test: {len(test_leads)}")
print(f"\nDistribution by Stage:")
print(test_leads['Stage'].value_counts(dropna=False))

Всего лидов с Source=Test: 159

Распределение по Stage:
Stage
Lost                    81
Call Delayed            73
Payment Done             3
Need to Call - Sales     1
Waiting For Payment      1
Name: count, dtype: int64


`Source = Test` rows don't look like test data → kept.

### Stage → Stage_Group + Won definition

Group Stage into Won / Lost / In Progress. Some `Payment Done` rows lack amount, course, or closing date — diagnose before defining Won.

In [28]:
won = deals[deals['Stage'] == 'Payment Done']
print(f"Total Payment Done: {len(won)}")
print()
print(f"Without Closing Date: {won['Closing Date'].isna().sum()}")
print(f"Without Initial Amount Paid: {won['Initial Amount Paid'].isna().sum()}")
print(f"Without Product: {won['Product'].isna().sum()}")
print(f"Without Months of study: {won['Months of study'].isna().sum()}")
print(f"Without Payment Type: {won['Payment Type'].isna().sum()}")

Всего Payment Done: 858

Без Closing Date: 337
Без Initial Amount Paid: 15
Без Product: 17
Без Months of study: 18
Без Payment Type: 494


18 rows: paid but never started studying; 15 rows: no amount (<2% of total). Not deleted — instead flagged: keep only clean data via a flag where Stage = Payment Done and Initial Amount Paid > 0.

In [29]:
stage_mapping = {  # group categories into Won — Payment Done, Lost — Lost, In Progress
    'Payment Done': 'Won',
    'Lost': 'Lost',
    'Call Delayed': 'In Progress',
    'Registered on Webinar': 'In Progress',
    'Waiting For Payment': 'In Progress',
    'Qualificated': 'In Progress',
    'Registered on Offline Day': 'In Progress',
    'Need to Call - Sales': 'In Progress',
    'Need To Call': 'In Progress',
    'Test Sent': 'In Progress',
    'Need a consultation': 'In Progress',
    'New Lead': 'In Progress',
    'Free Education': 'Won'
}

deals['Stage_Group'] = deals['Stage'].map(stage_mapping)

**`is_won_confirmed`** = Stage is `Payment Done` AND `Months of study` > 0. This is the business definition of a confirmed Won deal.

In [30]:

deals['is_won_confirmed'] = (
    (deals['Stage'] == 'Payment Done')
    & deals['Months of study'].notna()
    & (deals['Months of study'] > 0)
)
print(f"Won by Stage: {(deals['Stage_Group']=='Won').sum()}")
print(f"Won Confirmed (study started > 0): {deals['is_won_confirmed'].sum()}")
print(f"Difference: {(deals['Stage_Group']=='Won').sum() - deals['is_won_confirmed'].sum()} deals")

Won по Stage: 859
Won Confirmed (с началом обучения > 0): 839
Разница: 20 сделок


In [31]:
mask = (deals['Stage'] == 'Payment Done') & (deals['Months of study'] > 0) & (deals['Initial Amount Paid'] == 0)
deals.loc[mask, ['Id', 'Product', 'Course duration', 'Education Type',
                 'Payment Type', 'Offer Total Amount', 'Months of study', 'Initial Amount Paid']]

,Id,Product,Course duration,Education Type,Payment Type,Offer Total Amount,Months of study,Initial Amount Paid
21111,5805028000003130171,Digital Marketing,11.0,Morning,NaN,0.0,11.0,0.0


859 rows — one Free Education row was added to Won.

In [32]:
print(deals['Stage_Group'].value_counts(dropna=False))

Stage_Group
Lost           15743
In Progress     4991
Won              859
Name: count, dtype: int64


# !!!!!!

In [33]:
print(deals['Stage_Group'].isna().sum())

0


## 5. Payment field — fill missing values

In [34]:
won = deals[deals['is_won_confirmed']]
print(f"Total Won Confirmed: {len(won)}")
print(f"With Payment Type: {won['Payment Type'].notna().sum()}")
print(f"Without Payment Type (NaN): {won['Payment Type'].isna().sum()}")
print()
print("Offer distribution for Won without Payment Type:")
print(won[won['Payment Type'].isna()]['Offer Total Amount'].describe())

Всего Won Confirmed: 839
С Payment Type: 360
Без Payment Type (NaN): 479

Распределение Offer у Won без Payment Type:
count      479.000000
mean     10063.465553
std       2275.000872
min          0.000000
25%      11000.000000
50%      11000.000000
75%      11000.000000
max      11500.000000
Name: Offer Total Amount, dtype: float64


More than half of Won deals have no Payment Type, but we need it for unit economics. Let's see if we can classify them as one-time or monthly payment.

In [35]:
won_no_pt = deals[deals['is_won_confirmed'] & deals['Payment Type'].isna()].copy()
won_no_pt['ratio_rec'] = (won_no_pt['Initial Amount Paid'] * won_no_pt['Course duration']) / won_no_pt['Offer Total Amount']
won_no_pt['ratio_op'] = won_no_pt['Initial Amount Paid'] / won_no_pt['Offer Total Amount']

print("Deals that look like Recurring (ratio_rec in [0.7, 1.3]):")
print((won_no_pt['ratio_rec'].between(0.7, 1.3)).sum())

print("\nDeals that look like One Payment (ratio_op >= 0.8):")
print((won_no_pt['ratio_op'] >= 0.8).sum())

print("\nNeither:")
mask_unclear = ~won_no_pt['ratio_rec'].between(0.7, 1.3) & ~(won_no_pt['ratio_op'] >= 0.8)
print(mask_unclear.sum())

Сделки, похожие на Recurring (ratio_rec в [0.7, 1.3]):
450

Сделки, похожие на One Payment (ratio_op >= 0.8):
16

Ни туда, ни сюда:
13


In [36]:
# Look at the remaining 13 deals
won_no_pt = deals[deals['is_won_confirmed'] & deals['Payment Type'].isna()].copy()
won_no_pt['ratio_rec'] = (won_no_pt['Initial Amount Paid'] * won_no_pt['Course duration']) / won_no_pt['Offer Total Amount']
won_no_pt['ratio_op'] = won_no_pt['Initial Amount Paid'] / won_no_pt['Offer Total Amount']

mask_unclear = ~won_no_pt['ratio_rec'].between(0.7, 1.3) & ~(won_no_pt['ratio_op'] >= 0.8)

cols = ['Product', 'Course duration', 'Education Type', 'Months of study',
        'Initial Amount Paid', 'Offer Total Amount', 'ratio_op', 'ratio_rec']
won_no_pt.loc[mask_unclear, cols]

,Product,Course duration,Education Type,Months of study,Initial Amount Paid,Offer Total Amount,ratio_op,ratio_rec
497,Web Developer,6.0,Morning,1.0,1000.0,9000.0,0.111111,0.666667
833,Web Developer,6.0,Morning,1.0,1000.0,9000.0,0.111111,0.666667
1315,Web Developer,6.0,Morning,1.0,1000.0,9000.0,0.111111,0.666667
1625,Web Developer,6.0,Morning,1.0,1000.0,9000.0,0.111111,0.666667
1847,Web Developer,6.0,Morning,2.0,1000.0,9000.0,0.111111,0.666667
2159,Web Developer,6.0,Morning,2.0,1000.0,9000.0,0.111111,0.666667
2571,Web Developer,6.0,Morning,2.0,1000.0,9000.0,0.111111,0.666667
5575,Web Developer,6.0,Morning,3.0,1000.0,9000.0,0.111111,0.666667
6129,Web Developer,6.0,Morning,3.0,1000.0,9000.0,0.111111,0.666667
7406,Web Developer,6.0,Morning,4.0,1000.0,9000.0,0.111111,0.666667


In [37]:
extra_cols = ['Created Time', 'Closing Date', 'Deal Owner Name',
              'Source', 'Stage', 'Lost Reason']

cols_full = extra_cols + ['Product', 'Months of study',
                          'Initial Amount Paid', 'Offer Total Amount']

won_no_pt.loc[mask_unclear, cols_full]

,Created Time,Closing Date,Deal Owner Name,Source,Stage,Lost Reason,Product,Months of study,Initial Amount Paid,Offer Total Amount
497,2024-06-15 12:20:00,NaT,Ben Hall,Telegram posts,Payment Done,NaN,Web Developer,1.0,1000.0,9000.0
833,2024-06-11 19:39:00,NaT,Ben Hall,Organic,Payment Done,NaN,Web Developer,1.0,1000.0,9000.0
1315,2024-06-06 01:58:00,NaT,Eva Kent,Tiktok Ads,Payment Done,Stopped Answering,Web Developer,1.0,1000.0,9000.0
1625,2024-06-01 16:37:00,NaT,Quincy Vincent,SMM,Payment Done,NaN,Web Developer,1.0,1000.0,9000.0
1847,2024-05-30 00:56:00,NaT,Ben Hall,Youtube Ads,Payment Done,NaN,Web Developer,2.0,1000.0,9000.0
2159,2024-05-24 00:51:00,2024-05-27,Cara Iverson,Organic,Payment Done,Doesn't Answer,Web Developer,2.0,1000.0,9000.0
2571,2024-05-15 08:25:00,NaT,Paula Underwood,Youtube Ads,Payment Done,NaN,Web Developer,2.0,1000.0,9000.0
5575,2024-04-12 13:27:00,NaT,Charlie Davis,SMM,Payment Done,NaN,Web Developer,3.0,1000.0,9000.0
6129,2024-04-08 12:58:00,NaT,Victor Barnes,SMM,Payment Done,NaN,Web Developer,3.0,1000.0,9000.0
7406,2024-03-23 16:16:00,NaT,Paula Underwood,Tiktok Ads,Payment Done,NaN,Web Developer,4.0,1000.0,9000.0


Issue: 12 deals with a 9,000 offer and 6-month duration, plus one deal with no amount. The amount-less one is dropped (insignificant, and the sum can't be inferred — 5 possible variants). The rest are treated as monthly (recurring) payment.

In [38]:
# Mask: Won Confirmed without Payment Type
mask_won_no_pt = deals['is_won_confirmed'] & deals['Payment Type'].isna()

# Ratios
deals_ratio_rec = (deals['Initial Amount Paid'] * deals['Course duration']) / deals['Offer Total Amount']
deals_ratio_op = deals['Initial Amount Paid'] / deals['Offer Total Amount']

# Rule 1: Recurring — main pattern (450 deals 1000/11000) + 12 Web Developer 1000/9000
# Condition: ratio_rec in [0.5, 1.3] AND ratio_op < 0.8 (not a near-full single payment)

mask_recurring = mask_won_no_pt & deals_ratio_rec.between(0.5, 1.3) & (deals_ratio_op < 0.8)

# Rule 2: One Payment — almost full payment
mask_one_payment = mask_won_no_pt & (deals_ratio_op >= 0.8)

# Apply
deals.loc[mask_recurring, 'Payment Type'] = 'Recurring Payments'
deals.loc[mask_one_payment, 'Payment Type'] = 'One Payment'

print(f"Filled as Recurring Payments: {mask_recurring.sum()}")
print(f"Filled as One Payment: {mask_one_payment.sum()}")
print(f"Remaining NaN among Won Confirmed: {(deals['is_won_confirmed'] & deals['Payment Type'].isna()).sum()}")

Заполнено как Recurring Payments: 462
Заполнено как One Payment: 16
Осталось NaN среди Won Confirmed: 1


**ratio_op = Initial / Offer** — share of the one-time payment relative to the full course price. If ≈ 1, the person paid almost everything at once → One Payment.

**ratio_rec = (Initial × Course duration) / Offer** — if Initial were a monthly payment, multiplying it by course length should give the full price. If ratio_rec ≈ 1 → Recurring (monthly payment).

### Revenue column: actual money received as of the snapshot date

In [39]:
# === AOV_I and revenue_actual ===
# Single formula for all Won deals (Recurring and One Payment).
# Payment Type is the contractual form, not the real payment flow.
# AOV_I = (Fpay + (Months-1) * monthly_rest) / Months  — average monthly payment per student
# revenue_actual = AOV_I * Months, capped at Offer Total Amount

# Step 1 — monthly remainder
monthly_rest = (deals['Offer Total Amount'] - deals['Initial Amount Paid']) / (deals['Course duration'] - 1)

# Step 2 — revenue by formula (no cap)
revenue_raw = deals['Initial Amount Paid'] + (deals['Months of study'] - 1) * monthly_rest

# Step 3 — cap at Offer (what we actually recognize as revenue)
deals['revenue_actual'] = np.minimum(revenue_raw, deals['Offer Total Amount']).round(0)

# Step 4 — AOV_I = revenue_actual / Months (meaning: average payment per month of study)
deals['aov_i'] = np.where(
    deals['Months of study'] > 0,
    deals['revenue_actual'] / deals['Months of study'],
    0
).round(2)

# Step 5 — zero out non-Won
deals.loc[~deals['is_won_confirmed'], ['aov_i', 'revenue_actual']] = 0

In [40]:
won = deals['is_won_confirmed']
print(f"Won Confirmed: {won.sum()}")
print(f"Total revenue: €{deals.loc[won, 'revenue_actual'].sum():,.0f}")
print(f"Average AOV_I (school-wide): €{deals.loc[won, 'aov_i'].mean():,.2f}")
print(f"\nBy product:")
print(
    deals[won]
    .groupby('Product')
    .agg(
        deals=('Id', 'count'),
        revenue=('revenue_actual', 'sum'),
        avg_aov_i=('aov_i', 'mean'),
        total_months=('Months of study', 'sum')
    )
    .sort_values('revenue', ascending=False)
)

# Control: revenue does not exceed Offer for Won
overpaid = (deals.loc[won, 'revenue_actual'] > deals.loc[won, 'Offer Total Amount']).sum()
print(f"\nDeals where revenue > Offer (should be 0): {overpaid}")

Won Confirmed: 839
Общая выручка: €3,566,945
Средний AOV_I по школе: €870.99

По продуктам:
                   deals    revenue   avg_aov_i  total_months
Product                                                      
Digital Marketing    474  2251320.0  843.653544        2897.0
UX/UI Design         228   949045.0  953.277982        1170.0
Web Developer        137   366580.0  828.607810         505.0

Сделок где revenue > Offer (должно быть 0): 0


The code accounts for overpayments where a student paid more than the course price — there are 19 such cases in the dataset.

Three main products account for most paid deals (based on Excel filtering) — add a flag for them.

In [41]:
MAIN_PRODUCTS = ['Web Developer', 'Digital Marketing', 'UX/UI Design']
deals['is_main_product'] = deals['Product'].isin(MAIN_PRODUCTS)
print(f"Deals with a main product: {deals['is_main_product'].sum()}")
print(f"\nProduct distribution:")
print(deals['Product'].value_counts(dropna=False))

Сделок с основным продуктом: 3587

Распределение Product:
Product
NaN                    18001
Digital Marketing       1990
UX/UI Design            1022
Web Developer            575
Find yourself in IT        4
Data Analytics             1
Name: count, dtype: int64


### 6. Normalize German level to a single system

In [42]:
# Pattern: one letter (Latin A/B/C or Cyrillic А/В/С/Б) + digit 1 or 2

pattern = re.compile(r'([AaBbCcАаВвСсБб])\s*([12])')
translit = {
    'А': 'A', 'а': 'A',  # Cyr A -> Lat A
    'В': 'B', 'в': 'B',  # Cyr V -> Lat B (visually similar)
    'Б': 'B', 'б': 'B',  # Cyr B -> Lat B (phonetic)
    'С': 'C', 'с': 'C',  # Cyr S -> Lat C
}

In [43]:
def extract_deutsch_level(value):
    if pd.isna(value):
        return None
    s = str(value)
    match = pattern.search(s)
    if not match:
        return None
    letter, digit = match.group(1), match.group(2)
    # Transliterate if Cyrillic
    letter = translit.get(letter, letter)
    return f"{letter.upper()}{digit}"

deals['Level of Deutsch'] = deals['Level of Deutsch'].apply(extract_deutsch_level)

`pattern.search(s)` returns the first match in the string. This means:

- "B1-B2" -> B1 (lower level)
- "A2-B1 studying" -> A2 (lower)
- "B1 (already studying for B2)" -> B1 (current, not desired)
- "B2 exam in January" -> B2 (date not parsed, but lucky)
- "not sure about the level..." -> nothing found -> NaN

In [44]:
print(deals['Level of Deutsch'].value_counts(dropna=False))
print(f"\nValue types: {deals['Level of Deutsch'].nunique()}")

Level of Deutsch
NaN    20400
B1       817
B2       171
A2       149
C1        27
A1        26
C2         3
Name: count, dtype: int64

Типов значений: 6


58 values lost (1,251 -> 1,193) — 58 rows of free text with no explicit level.

In [45]:
deals['Deal Owner Name'] = deals['Deal Owner Name'].fillna('Unknown manager')  # fill missing manager with "Unknown manager"

## 7. Duplicate check

In [46]:
print(f"Duplicate Ids: {deals['Id'].duplicated().sum()}")
print(f"Fully duplicated rows: {deals.duplicated().sum()}")

Дубликатов по Id: 0
Полных дубликатов строк: 0


In [47]:
# How many deals each contact has
contact_counts = deals['Contact Name'].value_counts(dropna=False)
print(f"Distribution of deals per contact:")
print(contact_counts.value_counts().sort_index().head(15))

Распределение количества сделок на контакт:
count
1     15551
2      1990
3       385
4       113
5        28
6         9
7         4
8         2
10        2
11        1
13        1
19        1
39        1
54        1
61        1
Name: count, dtype: int64


In [48]:
# Top-10 contacts by number of deals
top_repeats = contact_counts.head(10)
print("Top-10 contacts by number of deals:")
print(top_repeats)

Топ-10 контактов с наибольшим количеством сделок:
Contact Name
NaN                    61
5805028000003014152    54
5805028000005448163    39
5805028000017522090    19
5805028000014478367    13
5805028000000872003    11
5805028000002150769    10
5805028000003966221    10
5805028000003456746     8
5805028000005124061     8
Name: count, dtype: int64


In [49]:
# Contacts with more than one paid deal (Stage_Group = Won)
won_per_contact = deals[deals['Stage_Group'] == 'Won']['Contact Name'].value_counts()
print(f"\nContacts with >1 paid deal:")
print((won_per_contact > 1).sum())

print(f"\nContacts with 2+ Won:")
print(won_per_contact[won_per_contact > 1].head(20))


Контактов с >1 оплаченной сделкой:
4

Контактов с 2+ Won:
Contact Name
5805028000043599528    2
5805028000009072093    2
5805028000033276868    2
5805028000017756471    2
Name: count, dtype: int64


In [50]:
# What is this contact with 54 deals?
suspicious_contact = '5805028000003014152'
investigation = deals[deals['Contact Name'] == suspicious_contact]
print(f"Deals: {len(investigation)}")
print(f"\nDeal Owners: ")
print(investigation['Deal Owner Name'].value_counts())
print(f"\nStage:")
print(investigation['Stage'].value_counts())
print(f"\nSource:")
print(investigation['Source'].value_counts())
print(f"\nCreated Time range:")
print(f"  from {investigation['Created Time'].min()}")
print(f"  to {investigation['Created Time'].max()}")

Сделок: 54

Deal Owners: 
Deal Owner Name
Julia Nelson     39
Diana Evans       8
Ian Miller        3
Bob Brown         2
Kevin Parker      1
Charlie Davis     1
Name: count, dtype: int64

Stage:
Stage
Lost    54
Name: count, dtype: int64

Source:
Source
Google Ads     31
Youtube Ads    19
CRM             2
Tiktok Ads      2
Name: count, dtype: int64

Диапазон Created Time:
  с 2023-07-25 11:05:00
  по 2024-05-30 22:26:00


### How many deals did contacts without a name make?

In [51]:
no_contact = deals[deals['Contact Name'].isna()]
print(no_contact['Stage_Group'].value_counts())

Stage_Group
Lost           43
Won            10
In Progress     8
Name: count, dtype: int64


In [52]:
# Other contacts with 6-54 deals
mid_repeats = contact_counts[(contact_counts >= 6) & (contact_counts <= 54)].index
print(f"Contacts with 6-54 deals: {len(mid_repeats)}")
print(f"Total deals for them: {deals['Contact Name'].isin(mid_repeats).sum()}")

# What are their stages?
mid_repeat_deals = deals[deals['Contact Name'].isin(mid_repeats)]
print(f"\nTheir Stage_Group:")
print(mid_repeat_deals['Stage_Group'].value_counts())

Контактов с 6-54 сделками: 22
Всего сделок у них: 254

Их Stage_Group:
Stage_Group
Lost           190
In Progress     61
Won              3
Name: count, dtype: int64


In [53]:
deals['is_duplicate_lead'] = deals['Lost Reason'] == 'Duplicate'
print(f"Duplicate leads: {deals['is_duplicate_lead'].sum()}")
print(f"Share of Lost: {deals['is_duplicate_lead'].sum() / (deals['Stage_Group'] == 'Lost').sum() * 100:.1f}%")

Дублирующих лидов: 1771
Доля от Lost: 11.2%


In [54]:
# Outlier = more than 5 deals AND all Lost (no Won, no In Progress at close)
contact_stats = deals.groupby('Contact Name').agg(
    total_deals=('Id', 'count'),
    won_deals=('Stage_Group', lambda x: (x == 'Won').sum())
)

outlier_contacts = contact_stats[
    (contact_stats['total_deals'] > 5) & 
    (contact_stats['won_deals'] == 0)
].index

deals['is_outlier_contact'] = deals['Contact Name'].isin(outlier_contacts)
print(f"Outlier contacts: {len(outlier_contacts)}")
print(f"Flagged deals: {deals['is_outlier_contact'].sum()}")

Outlier контактов: 19
Помечено сделок: 199


In [55]:
print(deals.shape)
print(deals.dtypes)

(21593, 30)
Id                                 str
Deal Owner Name                    str
Closing Date            datetime64[us]
Quality                            str
Stage                              str
Lost Reason                        str
Page                               str
Campaign                           str
SLA                    timedelta64[us]
Content                            str
Term                               str
Source                             str
Payment Type                       str
Product                            str
Education Type                     str
Created Time            datetime64[us]
Course duration                float64
Months of study                float64
Initial Amount Paid            float64
Offer Total Amount             float64
Contact Name                       str
City                               str
Level of Deutsch                   str
Stage_Group                        str
is_won_confirmed                  bool
revenue_actua

In [56]:
deals.shape


(21593, 30)

## 8. Date-range check

In [57]:
print("=" * 60)
print("FINAL SUMMARY — DEALS")
print("=" * 60)
print(f"Shape: {deals.shape}")
print(f"\nDate range:")
print(f"  Created Time:  {deals['Created Time'].min()} — {deals['Created Time'].max()}")
print(f"  Closing Date:  {deals['Closing Date'].min()} — {deals['Closing Date'].max()}")
print(f"\nConversion (Won/Won+Lost): {(deals['Stage_Group']=='Won').sum() / ((deals['Stage_Group']=='Won').sum() + (deals['Stage_Group']=='Lost').sum()) * 100:.2f}%")
print(f"\nFinance:")
print(f"  Total revenue (sum Initial Amount Paid): €{deals['Initial Amount Paid'].sum():,.0f}")
print(f"  Sum of all Offer Total: €{deals['Offer Total Amount'].sum():,.0f}")

ФИНАЛЬНАЯ СВОДКА — DEALS
Размер: (21593, 30)

Диапазон дат:
  Created Time:  2023-07-03 17:03:00 — 2024-06-21 15:30:00
  Closing Date:  2022-10-11 00:00:00 — 2024-12-11 00:00:00

Конверсия (Won/Won+Lost): 5.17%

Финансы:
  Общая выручка (sum Initial Amount Paid): €3,957,112
  Сумма всех Offer Total: €29,833,711


In [58]:
mask_invalid_order = (
    (deals['Closing Date'].dt.date < deals['Created Time'].dt.date) 
    & deals['Closing Date'].notna()
)

print(f"Anomalies found: {mask_invalid_order.sum()}")
print(deals.loc[mask_invalid_order, ['Id', 'Created Time', 'Closing Date', 'Stage', 'Deal Owner Name']])

Обнаружено аномалий: 44
                        Id        Created Time Closing Date         Stage  \
454    5805028000055502890 2024-06-16 00:06:00   2024-06-11          Lost   
2083   5805028000051847114 2024-05-25 21:29:00   2024-05-22          Lost   
2787   5805028000049539444 2024-05-12 11:19:00   2024-05-07          Lost   
3019   5805028000048886321 2024-05-08 15:31:00   2024-05-07  Payment Done   
3022   5805028000048883316 2024-05-08 14:48:00   2024-04-17          Lost   
3031   5805028000048886160 2024-05-08 12:54:00   2024-05-07  Payment Done   
3691   5805028000047482214 2024-04-30 15:16:00   2024-04-23          Lost   
4107   5805028000046234250 2024-04-24 17:30:00   2024-04-17          Lost   
4169   5805028000045961466 2024-04-23 21:44:00   2024-04-18          Lost   
4434   5805028000045314301 2024-04-21 08:57:00   2024-04-18          Lost   
4520   5805028000045245616 2024-04-20 06:19:00   2023-08-21          Lost   
4798   5805028000044499302 2024-04-17 09:10:00   202

### Found 44 anomalies where Created Time is later than Closing Date
Plus one likely typo — 2022-10-11 — outside the dataset range. Let's look inside these deals.

In [59]:
deals['is_closing_date_anomaly'] = (
    (deals['Closing Date'].dt.date < deals['Created Time'].dt.date) 
    & deals['Closing Date'].notna()
)

In [60]:
anomaly_won = deals[
    deals['is_closing_date_anomaly'] & 
    (deals['Stage'] == 'Payment Done')
]
print(anomaly_won[['Id', 'Created Time', 'Closing Date', 'Months of study', 
                   'Initial Amount Paid', 'Offer Total Amount', 
                   'Product', 'Course duration']])

                        Id        Created Time Closing Date  Months of study  \
3019   5805028000048886321 2024-05-08 15:31:00   2024-05-07              2.0   
3031   5805028000048886160 2024-05-08 12:54:00   2024-05-07              1.0   
6357   5805028000042015392 2024-04-05 10:40:00   2023-10-03              1.0   
14007  5805028000022036007 2023-12-19 10:37:00   2023-07-01              NaN   

       Initial Amount Paid  Offer Total Amount            Product  \
3019                1000.0             11000.0       UX/UI Design   
3031                1000.0             11000.0  Digital Marketing   
6357                1000.0             11000.0       UX/UI Design   
14007                  0.0                 0.0                NaN   

       Course duration  
3019              11.0  
3031              11.0  
6357              11.0  
14007              NaN  


## Lost Reason — exploration

In [61]:
# Lost Reason filled, but Stage is not Lost
non_lost_with_reason = deals[
    (deals['Lost Reason'].notna()) 
    & (deals['Stage'] != 'Lost')
]
print(f"Total: {len(non_lost_with_reason)}")
print(f"\nDistribution by Stage:")
print(non_lost_with_reason['Stage'].value_counts())
print(f"\nTop reasons in these rows:")
print(non_lost_with_reason['Lost Reason'].value_counts().head(15))

Всего: 428

Распределение по Stage:
Stage
Call Delayed                 257
Payment Done                 115
Waiting For Payment           30
Registered on Offline Day     15
Registered on Webinar          4
Qualificated                   3
Need to Call - Sales           2
Test Sent                      2
Name: count, dtype: int64

Топ причин в этих строках:
Lost Reason
Next stream                            157
Doesn't Answer                          61
needs time to think                     49
Stopped Answering                       32
Non target                              25
Duplicate                               25
Changed Decision                        24
Invalid number                          21
Expensive                               12
Gutstein refusal                         9
Conditions are not suitable              7
Inadequate                               2
Didn't leave an application              2
Does not know how to use a computer      1
Went to Rivals            

In [62]:
won_with_reason = deals[
    (deals['Lost Reason'].notna()) 
    & (deals['Stage'] == 'Payment Done')
]
print(f"Payment Done with Lost Reason: {len(won_with_reason)}")
print(won_with_reason['Lost Reason'].value_counts())

Payment Done с Lost Reason: 115
Lost Reason
Next stream                            40
Stopped Answering                      15
Doesn't Answer                         15
needs time to think                    15
Changed Decision                        7
Expensive                               6
Non target                              4
Conditions are not suitable             4
Gutstein refusal                        3
Invalid number                          3
Duplicate                               2
Does not know how to use a computer     1
Name: count, dtype: int64


**Lost Reason** serves several roles beyond a direct loss cause:

- Loss reason (Stage = Lost): ~5,040 cases
- Transfer to next stream (Next stream): 157 cases in Call Delayed, Payment Done, etc.
- Historical record of difficulties (Doesn't Answer, Stopped Answering, etc. with Stage ≠ Lost): ~110 cases — reflects past communication attempts.

Only deals with Stage = Lost are used in the loss-reason analysis.

3 deals are paid and their study is ongoing.

**Decision:** flag these 44 dates as anomalous, to exclude them from time-to-close calculation (so it doesn't go negative). For unit economics, rely on the closing date.

## 9. Save

In [ ]:
# deals.to_parquet(DATA_PROCESSED / 'deals_clean.parquet')
# deals.to_excel(DATA_PROCESSED / 'deals_clean.xlsx', index=False)

# print("Saved!")
# print(f"   {DATA_PROCESSED / 'deals_clean.parquet'}")
# print(f"   {DATA_PROCESSED / 'deals_clean.xlsx'}")

✅ Сохранено!
   ..\data\processed\deals_clean.parquet
   ..\data\processed\deals_clean.xlsx
